- SIRALAMA
- ERİŞİM
- ARALIKLANDIRMA
- GÖRSELLEŞTİRME

In [1]:
!pip install mesa[rec]
!pip install seaborn

# Has multi-dimensional arrays and matrices.
# Has a large collection of mathematical functions to operate on these arrays.
import numpy as np

# Data manipulation and analysis.
import pandas as pd

# Data visualization tools.
import seaborn as sns

import mesa



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.0/197.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.8/265.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.1 MB/s eta 0:00:00


In [19]:

from mesa.space import MultiGrid
from mesa.datacollection import DataCollector
import random

# Ajan sınıfını tanımlayalım
class MyAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.active = random.choice([True, False])
        self.wealth = random.randint(0, 100)
        self.group = "A" if self.unique_id % 2 == 0 else "B"  # Çift-ID'liler "A" grubu
        self.alis_fiyati = 0
        self.alis_miktari = 0



    def say_hi(self):
        print(
            f"Merhaba! Ben {self.unique_id}, "
            f"Servetim: {self.wealth}, "
            f"Aktif mi? {self.active}, "
            f"Grubum: {self.group}"
        )


    def step(self):
        # Ajanın her adımda yapacağı işlemler
        print(f"Ajan ID: {self.unique_id}- Servet: {self.wealth}")
        self.say_hi()

        self.alis_fiyati = random.randint(10, 100)
        self.alis_miktari = random.randint(1, 10)
        self.model.agent_list_al.append(self)


        self.satis_fiyati = random.randint(10, 100)
        self.satis_miktari = random.randint(1, 10)
        self.model.agent_list_sat.append(self)

        self.model.rich_agents.append(self)



class PiyasaYapiciAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.bakiye = 1000000  # Büyük başlangıç bakiyesi
        self.stok = 1000000    # Büyük başlangıç stok
        self.piysa_fiyati = 100  # Başlangıç piyasa fiyatı

    def step(self):
        # Toplam alış-satış miktarlarını hesapla
        toplam_alis = sum(agent.alis_miktari for agent in self.model.agent_list_al)
        toplam_satis = sum(agent.satis_miktari for agent in self.model.agent_list_sat)

        # Fazlalık hesapla
        fazlalik = abs(toplam_alis - toplam_satis)

        if toplam_alis > toplam_satis:
            self._fazlaligi_karsila('ALIS', fazlalik)
        elif toplam_satis > toplam_alis:
            self._fazlaligi_karsila('SATIS', fazlalik)

    def _fazlaligi_karsila(self, fazlalik_tipi, miktar):
        """Fazlalığı piyasa fiyatından karşılar"""
        if fazlalik_tipi == 'ALIS':
            # Satış fazlası varsa piyasa yapıcı alım yapar
            alim_miktari = min(miktar, self.bakiye / self.piysa_fiyati)
            self.stok += alim_miktari
            self.bakiye -= alim_miktari * self.piysa_fiyati
            print(f"Piyasa Yapıcı {alim_miktari:.2f} birim ALIM yaptı (Fiyat: {self.piysa_fiyati})")

            # Alış emirlerini güncelle (fazlalığı absorbe et)
            self._emir_azalt(self.model.agent_list_al, 'alis_miktari', alim_miktari)

        else:  # SATIS
            # Alış fazlası varsa piyasa yapıcı satış yapar
            satis_miktari = min(miktar, self.stok)
            self.stok -= satis_miktari
            self.bakiye += satis_miktari * self.piysa_fiyati
            print(f"Piyasa Yapıcı {satis_miktari:.2f} birim SATIŞ yaptı (Fiyat: {self.piysa_fiyati})")

            # Satış emirlerini güncelle
            self._emir_azalt(self.model.agent_list_sat, 'satis_miktari', satis_miktari)

    def _emir_azalt(self, agent_listesi, miktar_attr, azaltilacak_miktar):
        """Emirleri orantılı olarak azaltır"""
        toplam = sum(getattr(agent, miktar_attr) for agent in agent_listesi)

        for agent in agent_listesi:
            oran = getattr(agent, miktar_attr) / toplam
            azaltma = oran * azaltilacak_miktar
            yeni_miktar = max(0, getattr(agent, miktar_attr) - azaltma)
            setattr(agent, miktar_attr, yeni_miktar)



# Model sınıfını tanımlayalım
class MyModel(mesa.Model):
    def __init__(self, N, width, height):
        super().__init__()
        self.num_agents = N
        self.grid = MultiGrid(width, height, True)


        self.agent_list_al = []  # Ajanları saklamak için boş liste
        self.agent_list_sat = []  # Ajanları saklamak için boş liste

        self.rich_agents = []


        # Ajanları oluşturalım
        for i in range(self.num_agents):
            agent = MyAgent(self)
            self.agents.add(agent)

            # Ajanları rastgele bir hücreye yerleştirelim
            x = self.random.randrange(self.grid.width)
            y = self.random.randrange(self.grid.height)
            self.grid.place_agent(agent, (x, y))



        # Create and add the market maker
        self.piyasa_yapici = PiyasaYapiciAgent(self)
        self.agents.add(self.piyasa_yapici)

        # Veri toplamak için DataCollector kullanabiliriz (opsiyonel)
        self.datacollector = DataCollector()

    # Modelde filtreleme:
    def get_group_a(self):
        return self.agents.select(lambda agent: agent.group == "A")


    # Dinamik Ajan Ekleyip Çıkarma

    def dynamic_agents(self):
        # diğer piyasalara ilişkin kötü haberler arttığında ve mevcut piyasaya ilşkin iyi haberler arttığında, piyasaya ajan girişi diğer zamanlardaki
        # girişlerden daha çok olacaktır. Bu nedenle piyasaya gelen haberlere göre ajan girişi olacak
        if random.random() > 0.5: #iyi haber geldi, 100 yeni ajan girdi
            # Yeni ajan ekle
            num_new_agents = int(0.25 * len(self.agents))  # Calculate before loop
            for i in range(num_new_agents):   # yeni giriş yapan ajanların sayısını, mevcut ajan sayısının belli bir oranı olarak ayarlanabilir.
                new_agent = MyAgent(self)
                self.agents.add(new_agent)
            print(f"Yeni eklenen ajan SAYISI: {num_new_agents}")
        else:                    # iyi haber yok, 10 yeni ajan girdi
            # Yeni ajan ekle
            num_new_agents = int(0.10 * len(self.agents))
            for i in range(num_new_agents):
                new_agent = MyAgent(self)
                self.agents.add(new_agent)
            print(f"Yeni eklenen ajan sayısİ: {num_new_agents}")

        print(f"Güncel Ajan SAyısı: {len(self.agents)}")

        """
        # 5 ID'li ajanı sil
        agent_to_remove = next(a for a in self.agents if a.unique_id == 5)
        self.agents.remove(agent_to_remove)
        """




    def step(self):
        # Modelin her adımda yapacağı işlemler
        self.agents.do("step")
        self.dynamic_agents()

        self.datacollector.collect(self)

# Modeli oluşturalım (100 ajan, 10x10 grid)
model = MyModel(10, 10, 10)

# Tüm ajanları AgentSet olarak almak için:
all_agents = model.agents

# AgentSet'i kontrol edelim
print(f"Toplam ajan sayısı: {len(all_agents)}")
print(f"Piyasa Yapıcı ajanın ID'si: {all_agents[model.num_agents].unique_id}")

# Modeli birkaç adım çalıştıralım
for i in range(10):
    print(f"------Adım {i+1}--------")
    model.step()

Toplam ajan sayısı: 11
Piyasa Yapıcı ajanın ID'si: 11
------Adım 1--------
Ajan ID: 1- Servet: 63
Merhaba! Ben 1, Servetim: 63, Aktif mi? True, Grubum: B
Ajan ID: 2- Servet: 74
Merhaba! Ben 2, Servetim: 74, Aktif mi? True, Grubum: A
Ajan ID: 3- Servet: 60
Merhaba! Ben 3, Servetim: 60, Aktif mi? False, Grubum: B
Ajan ID: 4- Servet: 36
Merhaba! Ben 4, Servetim: 36, Aktif mi? False, Grubum: A
Ajan ID: 5- Servet: 3
Merhaba! Ben 5, Servetim: 3, Aktif mi? True, Grubum: B
Ajan ID: 6- Servet: 76
Merhaba! Ben 6, Servetim: 76, Aktif mi? False, Grubum: A
Ajan ID: 7- Servet: 73
Merhaba! Ben 7, Servetim: 73, Aktif mi? False, Grubum: B
Ajan ID: 8- Servet: 99
Merhaba! Ben 8, Servetim: 99, Aktif mi? False, Grubum: A
Ajan ID: 9- Servet: 9
Merhaba! Ben 9, Servetim: 9, Aktif mi? False, Grubum: B
Ajan ID: 10- Servet: 1
Merhaba! Ben 10, Servetim: 1, Aktif mi? True, Grubum: A
Piyasa Yapıcı 5.00 birim SATIŞ yaptı (Fiyat: 100)
Yeni eklenen ajan sayısİ: 1
Güncel Ajan SAyısı: 12
------Adım 2--------
Ajan ID: 1-

KAPALI PİYASA VS AÇIK PİYASA

- Eğer Açık Piyasa analizi yapılacaksa, yani sonradan piyasaya ajan girişi olacaksa tüm ajanların başlangıç varlık sayısı sıfır olmalıdır.

- Piyasa yapıcı aktif strateji izlerse, pasif strateji izlerse

- Piyasa yapıcı önce emir verirse (piyasayı yönlendirirse) , sonra emir verirse (piyasaya müdahale etmezse)

soru: python mesada bir piyasa yapıcı ajanı tanımlamak istiyorum, ister alış tarafında fazlalık olsun ister satış tarafında fazlalık olsun, fazlalık varsa piyasa yapıcı bu fazlalığı alsın ya da satsın. alış ya da satış fiyatı da piyasa fiyatı değerinde olsun

In [10]:
# İşte piyasa yapıcı (market maker) agent'ını Mesa'ya entegre eden kapsamlı bir çözüm:

class PiyasaYapiciAgent(mesa.Agent):
    def __init__(self, unique_id, model):
        super().__init__(unique_id, model)
        self.bakiye = 1000000  # Büyük başlangıç bakiyesi
        self.stok = 1000000    # Büyük başlangıç stok
        self.piysa_fiyati = 100  # Başlangıç piyasa fiyatı

    def step(self):
        # Toplam alış-satış miktarlarını hesapla
        toplam_alis = sum(agent.alis_miktari for agent in self.model.agent_list_al)
        toplam_satis = sum(agent.satis_miktari for agent in self.model.agent_list_sat)

        # Fazlalık hesapla
        fazlalik = abs(toplam_alis - toplam_satis)

        if toplam_alis > toplam_satis:
            self._fazlaligi_karsila('ALIS', fazlalik)
        elif toplam_satis > toplam_alis:
            self._fazlaligi_karsila('SATIS', fazlalik)

    def _fazlaligi_karsila(self, fazlalik_tipi, miktar):
        """Fazlalığı piyasa fiyatından karşılar"""
        if fazlalik_tipi == 'ALIS':
            # Satış fazlası varsa piyasa yapıcı alım yapar
            alim_miktari = min(miktar, self.bakiye / self.piysa_fiyati)
            self.stok += alim_miktari
            self.bakiye -= alim_miktari * self.piysa_fiyati
            print(f"Piyasa Yapıcı {alim_miktari:.2f} birim ALIM yaptı (Fiyat: {self.piysa_fiyati})")

            # Alış emirlerini güncelle (fazlalığı absorbe et)
            self._emir_azalt(self.model.agent_list_al, 'alis_miktari', alim_miktari)

        else:  # SATIS
            # Alış fazlası varsa piyasa yapıcı satış yapar
            satis_miktari = min(miktar, self.stok)
            self.stok -= satis_miktari
            self.bakiye += satis_miktari * self.piysa_fiyati
            print(f"Piyasa Yapıcı {satis_miktari:.2f} birim SATIŞ yaptı (Fiyat: {self.piysa_fiyati})")

            # Satış emirlerini güncelle
            self._emir_azalt(self.model.agent_list_sat, 'satis_miktari', satis_miktari)

    def _emir_azalt(self, agent_listesi, miktar_attr, azaltilacak_miktar):
        """Emirleri orantılı olarak azaltır"""
        toplam = sum(getattr(agent, miktar_attr) for agent in agent_listesi)

        for agent in agent_listesi:
            oran = getattr(agent, miktar_attr) / toplam
            azaltma = oran * azaltilacak_miktar
            yeni_miktar = max(0, getattr(agent, miktar_attr) - azaltma)
            setattr(agent, miktar_attr, yeni_miktar)

""" Modelinize Entegrasyon:"""

class PiyasaModeli(mesa.Model):
    def __init__(self, N):
        # Diğer init işlemleri...
        self.piyasa_yapici = PiyasaYapiciAgent(-1, self)  # Özel ID (-1)
        self.schedule.add(self.piyasa_yapici)

    def step(self):
        # Önce normal agentlar
        self.schedule.step()

        # Sonra piyasa yapıcı
        for agent in self.schedule.agents:
            if isinstance(agent, PiyasaYapiciAgent):
                agent.step()


SyntaxError: expected ':' (<ipython-input-10-9097d0674051>, line 3)

Çalışma Mantığı:
Dengeleme: Piyasa yapıcı her adımda alış-satış dengesizliğini kontrol eder

Fiyat Katılığı: İşlemleri sabit piyasa fiyatından yapar

Risk Sınırları:

Alım yaparken bakiyesini kontrol eder

Satım yaparken stok durumunu kontrol eder

Orantılı Azaltma: Fazlalığı tüm emirlere orantılı dağıtır

In [ ]:
# Gelişmiş Özellikler Eklemek İçin:


# Dinamik fiyat ayarı
def fiyat_guncelle(self):
    spread = 0.01  # %1 spread
    self.alis_fiyati = self.piysa_fiyati * (1 - spread)
    self.satis_fiyati = self.piysa_fiyati * (1 + spread)

# Stok-bakiye koruması
def _guvenli_miktar(self, miktar, islem_tipi):
    if islem_tipi == 'ALIS':
        return min(miktar, self.bakiye / self.satis_fiyati)
    else:
        return min(miktar, self.stok)

Bu yapı sayesinde:

Piyasada likidite sağlanır

Aşırı fiyat oynaklığı önlenir

Dengeleme otomatik olarak yapılır

yukarıdaki kod, PiyasaModeli (piyasa modeli) sınıfındaki PiyasaYapiciAgent (piyasa yapıcı aracı) üzerinde adım yöntemini çağırmaya çalışıyor. Ancak, PiyasaModeli kod parçacığında tam olarak uygulanmamış ve bu da sorunlara neden olabilir.

Eksik Başlatma: PiyasaModeli sınıfının eksiksiz bir __init__ yöntemi yok. "Diğer init işlemleri..."nden bahsediyor ancak schedule niteliği gibi gerekli bileşenleri gerçekten ayarlamıyor. self.schedule düzgün bir şekilde başlatılmalı ve aracılar buna eklenmelidir.
Yanlış Zamanlama: Adım yönteminde piyasa yapıcı aracının çağrılma şekli Mesa içinde en verimli veya deyimsel yol olmayabilir. Mesa genellikle aracı aktivasyonunu yönetmek için bir schedule nesnesi kullanır. Kod, piyasa yapıcıyı bulmak için aracılar arasında manuel olarak yineleme yapar.

In [ ]:
from mesa import Model, Agent
from mesa.time import RandomActivation

class PiyasaYapiciAgent(Agent):
    # ... (rest of the PiyasaYapiciAgent code remains the same) ...

class PiyasaModeli(Model):
    def __init__(self, N):
        super().__init__()  # Call the parent class's __init__
        self.num_agents = N
        self.schedule = RandomActivation(self)  # Initialize the schedule
        self.agent_list_al = []
        self.agent_list_sat = []
        # Create agents and add them to the schedule
        for i in range(self.num_agents):
            agent = MyAgent(self)  # Assuming MyAgent is defined elsewhere
            self.schedule.add(agent)
        # Create and add the market maker
        self.piyasa_yapici = PiyasaYapiciAgent(-1, self)
        self.schedule.add(self.piyasa_yapici)

    def step(self):
        self.schedule.step()  # Activate all agents in the schedule

In [ ]:
# ajanların unique_id lerini ekrana yazdırmak istediğimizde piyasa yapıcının da getirir mi?
# Get all agents in the model
all_agents = model.schedule.agents

# Print unique_id for all agents, including the market maker
for agent in all_agents:
    print(f"Agent ID: {agent.unique_id}")


In [ ]:
# piyasa yapıcının unique_id sini almak istemiyorsak

# Get all agents in the model
all_agents = model.schedule.agents

# Print unique_id for agents, excluding the market maker
for agent in all_agents:
    if not isinstance(agent, PiyasaYapiciAgent):
        print(f"Agent ID: {agent.unique_id}")


In [ ]:
# diyelim ki tüm ajanları bir listeye almak istiyorum ve piyasa yapıcıyı bu listeye almak istemiyorum
# Bu, ajanların türlerine göre filtrelenmesini gerektirir.

# Get all agents in the model
all_agents = model.schedule.agents

# Create a list of agents, excluding the market maker
agent_list_without_market_maker = [agent for agent in all_agents if not isinstance(agent, PiyasaYapiciAgent)]

# Print the list to verify
print(agent_list_without_market_maker)

"""
Artık, piyasa yapıcısı hariç tüm aracılarla çalışmanız gereken herhangi bir işlem için
kullanabileceğiniz agent_list_without_market_maker listesine sahipsiniz. Örneğin,
her bir aracının özelliklerine erişmek veya yöntemlerini çağırmak için bu liste üzerinde
yineleme yapabilirsiniz:
"""

for agent in agent_list_without_market_maker:
    print(f"Agent ID: {agent.unique_id}, Wealth: {agent.wealth}")  # Assuming 'wealth' is an attribute of your agents